# 03 Peak Consolidation
Consolidate Runx3 and Runx1 peaks across day 5 and day 8 samples.

## 03.01 Initialize Environment
Load programs and start at working directory.

In [2]:
# Define a docker run function to simplify running docker commands
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

# Change to working directory
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
echo Working directory: $(pwd)
echo Files:
ls

bedtools v2.31.1
Working directory: /home/dalbao/AlbaoRunx3Manuscript/cutnrun
Files:
01_diffbind	   02_peakqc	    03_peakconsolidation	source_data
01_diffbind.ipynb  02_peakqc.ipynb  03_peakconsolidation.ipynb


## 03.02 Starting Data
Check starting data from steps 01 and 02.

In [3]:
# Differential calls
echo [step 01] Differential calls:
wc -l 01_diffbind/bdgdiff/*c5.0*

# QC-ed peak calls
echo [Step 02] QC-ed peak calls:
wc -l 02_peakqc/*above_inflection*.bed

[step 01] Differential calls:
    363 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_common.bed
  15898 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond1.bed
  27469 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond2.bed
   1254 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_common.bed
  43375 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond1.bed
  26721 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond2.bed
 115080 total
[Step 02] QC-ed peak calls:
   1002 02_peakqc/early_Runx1.merged.macs2_peaks.above_inflection.bed
    619 02_peakqc/early_Runx3.merged.macs2_peaks.above_inflection.bed
   1178 02_peakqc/late_Runx1.merged.macs2_peaks.above_inflection.bed
    921 02_peakqc/late_Runx3.merged.macs2_peaks.above_inflection.bed
   1728 02_peakqc/memory_Runx1.merged.macs2_peaks.above_inflection.bed
   1218 02_peakqc/memory_Runx3.merged.macs2_peaks.above_inflection.bed
   3305 02_peakqc/shCd19_Runx1.merged.macs2_peaks.above_inflection.bed
   2887 02_peakqc/shCd19_Runx3.me

## 03.03 Conseunsus Peaks
Create consensus peak files. Just union BED files as these are assumed to be **high-quality** peaks after step 02.
Make a day 5 Runx3 set from shCd19 and shRunx3, and a day 8 set from early, late, terminal and memory.

In [4]:
mkdir -p 03_peakconsolidation

INDIR=02_peakqc
OUTDIR=03_peakconsolidation

# Union high-quality peaks across samples into a consensus BED (chrom, start, end)
consensus() {
    local outname=$1
    shift
    cat "$@" \
        | sort -k1,1 -k2,2n \
        | bedtools merge -i - \
        > ${OUTDIR}/${outname}.consensus.bed
    echo ${outname}: $(wc -l < ${OUTDIR}/${outname}.consensus.bed) consensus peaks
}

# Day 5: shCd19 (control) and shRunx3 knockdown
consensus day5_Runx1 ${INDIR}/shCd19_Runx1.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/shRunx3_Runx1.merged.macs2_peaks.above_inflection.bed
consensus day5_Runx3 ${INDIR}/shCd19_Runx3.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/shRunx3_Runx3.merged.macs2_peaks.above_inflection.bed

# Day 8: early, late, terminal and memory differentiation stages
consensus day8_Runx1 ${INDIR}/early_Runx1.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/late_Runx1.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/terminal_Runx1.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/memory_Runx1.merged.macs2_peaks.above_inflection.bed
consensus day8_Runx3 ${INDIR}/early_Runx3.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/late_Runx3.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/terminal_Runx3.merged.macs2_peaks.above_inflection.bed \
    ${INDIR}/memory_Runx3.merged.macs2_peaks.above_inflection.bed

echo
echo Consensus files:
wc -l  ${OUTDIR}/*

day5_Runx1: 6757 consensus peaks
day5_Runx3: 5325 consensus peaks
day8_Runx1: 7141 consensus peaks
day8_Runx3: 3919 consensus peaks

Consensus files:
  6757 03_peakconsolidation/day5_Runx1.consensus.bed
  5325 03_peakconsolidation/day5_Runx3.consensus.bed
  7141 03_peakconsolidation/day8_Runx1.consensus.bed
  3919 03_peakconsolidation/day8_Runx3.consensus.bed
 23142 total


In [5]:
# Check overlap between Runx1 and Runx3 consensus peaks
echo
echo Day 5 Runx overlap
bedtools intersect -a ${OUTDIR}/day5_Runx1.consensus.bed -b ${OUTDIR}/day5_Runx3.consensus.bed -wa -wb \
    | sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    > ${OUTDIR}/day5_Runx1_Runx3.overlap.bed

wc -l ${OUTDIR}/day5_Runx1_Runx3.overlap.bed

echo
echo Day 8 Runx overlap
bedtools intersect -a ${OUTDIR}/day8_Runx1.consensus.bed -b ${OUTDIR}/day8_Runx3.consensus.bed -wa -wb \
    | sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    > ${OUTDIR}/day8_Runx1_Runx3.overlap.bed


wc -l ${OUTDIR}/day8_Runx1_Runx3.overlap.bed


Day 5 Runx overlap
677 03_peakconsolidation/day5_Runx1_Runx3.overlap.bed

Day 8 Runx overlap
39 03_peakconsolidation/day8_Runx1_Runx3.overlap.bed


In [6]:
# Merge Runx1 and Runx3 consensus peaks into one union
echo
echo Day 5 Runx3 and Runx1 union
cat ${OUTDIR}/day5_Runx1.consensus.bed ${OUTDIR}/day5_Runx3.consensus.bed \
    | sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    > ${OUTDIR}/day5_Runx1_Runx3.union.bed

echo Day 8 Runx3 and Runx1 union
cat ${OUTDIR}/day8_Runx1.consensus.bed ${OUTDIR}/day8_Runx3.consensus.bed \
    | sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    > ${OUTDIR}/day8_Runx1_Runx3.union.bed

wc -l ${OUTDIR}/*union*


Day 5 Runx3 and Runx1 union
Day 8 Runx3 and Runx1 union
 11402 03_peakconsolidation/day5_Runx1_Runx3.union.bed
 11021 03_peakconsolidation/day8_Runx1_Runx3.union.bed
 22423 total


## 03.04 RUNX3-bound Peaks

In [7]:
ls -1 ${OUTDIR}

day5_Runx1.consensus.bed
day5_Runx1_Runx3.overlap.bed
day5_Runx1_Runx3.union.bed
day5_Runx3.consensus.bed
day8_Runx1.consensus.bed
day8_Runx1_Runx3.overlap.bed
day8_Runx1_Runx3.union.bed
day8_Runx3.consensus.bed


Define Runx3-bound peaks as peaks in `day5_Runx3.consensus.bed` or `day8_Runx3.consensus.bed` that overlap with peaks in `01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond1.bed`.

Identify those peaks as `*_Runx3.bound.bed`

In [8]:
INDIFF=01_diffbind/bdgdiff
DIFFPEAKS=Runx3_shCd19_vs_shRunx3_c5.0_cond1.bed

# Strip the bdgdiff UCSC track header line
tail -n +2 ${INDIFF}/${DIFFPEAKS} > ${OUTDIR}/${DIFFPEAKS}.noheader.bed

for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx3.consensus.bed \
        -b ${OUTDIR}/${DIFFPEAKS}.noheader.bed \
        -u \
    > ${OUTDIR}/${day}_Runx3.bound.bed
    echo ${day} Runx3-bound peaks: $(wc -l < ${OUTDIR}/${day}_Runx3.bound.bed)
done

rm ${OUTDIR}/${DIFFPEAKS}.noheader.bed

day5 Runx3-bound peaks: 2360
day8 Runx3-bound peaks: 227


What is the intersection of `*_Runx1_Runx3.overlap.bed` to the Runx3-bound peaks?

In [9]:
echo Overlap Day 5 Runx1-Runx3-overlap to Runx3-bound peaks:
bedtools intersect \
    -a ${OUTDIR}/day5_Runx1_Runx3.overlap.bed \
    -b ${OUTDIR}/day5_Runx3.bound.bed \
    -u \
    | wc -l 

echo Overlap Day 8 Runx1-Runx3-overlap to Runx3-bound peaks:
bedtools intersect \
    -a ${OUTDIR}/day8_Runx1_Runx3.overlap.bed \
    -b ${OUTDIR}/day8_Runx3.bound.bed \
    -u \
    | wc -l

Overlap Day 5 Runx1-Runx3-overlap to Runx3-bound peaks:
492
Overlap Day 8 Runx1-Runx3-overlap to Runx3-bound peaks:
31


## 03.05 RUNX3-unknown and -constant peaks
There are Runx3 peaks that go up in shRunx3 defined in `01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond2.bed`. Check their overlap with `day5_Runx3.consensus.bed` or `day8_Runx3.consensus.bed`

Identify those peaks as `*_Runx3.unknown.bed`

In [10]:
INDIFF=01_diffbind/bdgdiff
DIFFPEAKS=Runx3_shCd19_vs_shRunx3_c5.0_cond2.bed

# Strip the bdgdiff UCSC track header line
tail -n +2 ${INDIFF}/${DIFFPEAKS} > ${OUTDIR}/${DIFFPEAKS}.noheader.bed

for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx3.consensus.bed \
        -b ${OUTDIR}/${DIFFPEAKS}.noheader.bed \
        -u \
    > ${OUTDIR}/${day}_Runx3.unknown.bed
    echo ${day} Runx3-unknown peaks: $(wc -l < ${OUTDIR}/${day}_Runx3.unknown.bed)
done

rm ${OUTDIR}/${DIFFPEAKS}.noheader.bed

day5 Runx3-unknown peaks: 1716
day8 Runx3-unknown peaks: 30


Consensus peaks not in `unknown` or `bound` peaks are considered `constant` peaks.

In [11]:
for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx3.consensus.bed \
        -b ${OUTDIR}/${day}_Runx3.unknown.bed ${OUTDIR}/${day}_Runx3.bound.bed \
        -v \
    > ${OUTDIR}/${day}_Runx3.constant.bed
    echo ${day} Runx3-constant peaks: $(wc -l < ${OUTDIR}/${day}_Runx3.constant.bed)
done

day5 Runx3-constant peaks: 1266
day8 Runx3-constant peaks: 3663


# 03.06 RUNX1-bound, -unknown or -constant peaks

Define Runx1-bound peaks as peaks in `day5_Runx1.consensus.bed` or `day8_Runx1.consensus.bed` that overlap with peaks in `01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond2.bed`.

Identify those peaks as `*_Runx1.bound.bed`

In [12]:
INDIFF=01_diffbind/bdgdiff
DIFFPEAKS=Runx1_shCd19_vs_shRunx3_c5.0_cond2.bed

# Strip the bdgdiff UCSC track header line
tail -n +2 ${INDIFF}/${DIFFPEAKS} > ${OUTDIR}/${DIFFPEAKS}.noheader.bed

for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx1.consensus.bed \
        -b ${OUTDIR}/${DIFFPEAKS}.noheader.bed \
        -u \
    > ${OUTDIR}/${day}_Runx1.bound.bed
    echo ${day} Runx1-bound peaks: $(wc -l < ${OUTDIR}/${day}_Runx1.bound.bed)
done

rm ${OUTDIR}/${DIFFPEAKS}.noheader.bed

day5 Runx1-bound peaks: 2482
day8 Runx1-bound peaks: 58


There are Runx1 peaks that go down in shRunx3 defined in `01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond1.bed`. Check their overlap with `day5_Runx1.consensus.bed` or `day8_Runx1.consensus.bed`

Identify those peaks as `*_Runx1.unknown.bed`

In [13]:
INDIFF=01_diffbind/bdgdiff
DIFFPEAKS=Runx1_shCd19_vs_shRunx3_c5.0_cond1.bed

# Strip the bdgdiff UCSC track header line
tail -n +2 ${INDIFF}/${DIFFPEAKS} > ${OUTDIR}/${DIFFPEAKS}.noheader.bed

for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx1.consensus.bed \
        -b ${OUTDIR}/${DIFFPEAKS}.noheader.bed \
        -u \
    > ${OUTDIR}/${day}_Runx1.unknown.bed
    echo ${day} Runx1-unknown peaks: $(wc -l < ${OUTDIR}/${day}_Runx1.unknown.bed)
done

rm ${OUTDIR}/${DIFFPEAKS}.noheader.bed

day5 Runx1-unknown peaks: 2067
day8 Runx1-unknown peaks: 32


Consensus peaks not in `unknown` or `bound` peaks are considered `constant` peaks

In [14]:
for day in day5 day8; do
    bedtools intersect \
        -a ${OUTDIR}/${day}_Runx1.consensus.bed \
        -b ${OUTDIR}/${day}_Runx1.unknown.bed ${OUTDIR}/${day}_Runx1.bound.bed \
        -v \
    > ${OUTDIR}/${day}_Runx1.constant.bed
    echo ${day} Runx1-constant peaks: $(wc -l < ${OUTDIR}/${day}_Runx1.constant.bed)
done

day5 Runx1-constant peaks: 2214
day8 Runx1-constant peaks: 7052


What is the distinct overlap of `day5_Runx1_Runx3.overlap.bed` to `day5_Runx1.bound.bed`, `day5_Runx1.unknown.bed`, and `day5_Runx1.constant.bed`?

In [16]:
echo Overlap Day 5 Runx1-Runx3-overlap to Runx1-bound peaks:
bedtools intersect \
    -a ${OUTDIR}/day5_Runx1_Runx3.overlap.bed \
    -b ${OUTDIR}/day5_Runx1.bound.bed \
    -u \
    | wc -l

echo Overlap Day 5 Runx1-Runx3-overlap to Runx1-unknown peaks:
bedtools intersect \
    -a ${OUTDIR}/day5_Runx1_Runx3.overlap.bed \
    -b ${OUTDIR}/day5_Runx1.unknown.bed \
    -u \
    | wc -l

echo Overlap Day 5 Runx1-Runx3-overlap to Runx1-constant peaks:
bedtools intersect \
    -a ${OUTDIR}/day5_Runx1_Runx3.overlap.bed \
    -b ${OUTDIR}/day5_Runx1.constant.bed \
    -u \
    | wc -l

Overlap Day 5 Runx1-Runx3-overlap to Runx1-bound peaks:
140
Overlap Day 5 Runx1-Runx3-overlap to Runx1-unknown peaks:
86
Overlap Day 5 Runx1-Runx3-overlap to Runx1-constant peaks:
455
